# RSNA Knee Abnormality Detection — Exploratory Data Analysis

This notebook presents an aggregate view of the competition's study labels,
MRI series, report characteristics, and slice-count distribution. It never
displays report text or study identifiers.

This dataset snapshot contains **4,407 training studies**, of which
only **58** have complete human labels. That small labeled sample motivates
careful internal cross-validation and restrained claims rather than treating
the results as independent validation.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

SEED = 42
IS_KAGGLE = Path("/kaggle/input").exists()
if not IS_KAGGLE:
    raise RuntimeError("This notebook runs on Kaggle only.")

DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
package_initializers = tuple(
    Path("/kaggle/input/datasets").rglob("knee_mri/__init__.py")
)
if len(package_initializers) != 1:
    raise RuntimeError("Expected exactly one attached knee_mri source package.")
sys.path.insert(0, str(package_initializers[0].parent.parent))

## 1. Competition Data Overview

In [ ]:
from knee_mri.dataset import split_labeled_studies
from knee_mri.labels import LABEL_COLUMNS
from knee_mri.weak_label_evaluation import orthographic_bucket

train_df = pd.read_csv(DATA_DIR / "train.csv")
series_df = pd.read_csv(DATA_DIR / "train_series.csv")
labeled, unlabeled = split_labeled_studies(train_df)

overview = pd.Series(
    {
        "Training Studies": len(train_df),
        "Human-Labeled Studies": len(labeled),
        "Report-Only Studies": len(unlabeled),
        "MRI Series": len(series_df),
    },
    name="Count",
).to_frame()
display(overview)

**Interpretation.** This run contains 4,407 training studies
and 24,371 MRI series, but only 58 studies (1.3%) have all 12 human targets.
The remaining 4,349 studies are report-only. Phase 3A therefore trains only
on the human-labeled subset and reports internal cross-validation rather than
claiming an independent confirmation set.

## 2. Dataset Schema and Representative Protocols

In [ ]:
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_df = pd.read_csv(DATA_DIR / "sample_submission.csv")
test_series_df = pd.read_csv(DATA_DIR / "test_series.csv")

file_shapes = pd.DataFrame(
    {
        "Rows": [
            len(train_df),
            len(test_df),
            len(sample_df),
            len(series_df),
            len(test_series_df),
        ],
        "Columns": [
            train_df.shape[1],
            test_df.shape[1],
            sample_df.shape[1],
            series_df.shape[1],
            test_series_df.shape[1],
        ],
    },
    index=[
        "train.csv",
        "test.csv",
        "sample_submission.csv",
        "train_series.csv",
        "test_series.csv",
    ],
)

train_schema = train_df.dtypes.astype(str).rename("Data Type").to_frame()

column_glossary = pd.DataFrame(
    [
        {
            "Column": "StudyInstanceUID",
            "Role": "Study identifier, shared by train.csv and test.csv",
            "Shown Here": "Never (identifier)",
        },
        {
            "Column": "Report",
            "Role": (
                "Free-text radiology report (multilingual); present in "
                "both train.csv and test.csv"
            ),
            "Shown Here": "Aggregate length and character-set evidence only",
        },
        {
            "Column": (
                f"{len(LABEL_COLUMNS)} target columns "
                f"({LABEL_COLUMNS[0]}, {LABEL_COLUMNS[1]}, ...)"
            ),
            "Role": (
                "Human-annotated finding presence (0/1); train.csv only, "
                "complete for 58 studies"
            ),
            "Shown Here": "Aggregate prevalence only (Section 3), never per-study",
        },
    ]
).set_index("Column")

# Aggregate series counts per observed acquisition-protocol combination --
# no individual series or study identifier is shown, only group counts.
protocol_combinations = (
    series_df.groupby(["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"])
    .size()
    .rename("Series")
    .to_frame()
)
protocol_combinations["Share"] = (
    protocol_combinations["Series"] / protocol_combinations["Series"].sum()
)

display(file_shapes)
display(train_schema)
display(column_glossary)
display(protocol_combinations)

**Interpretation.** `train.csv` contains report text plus the 12 training targets; `test.csv` shares the same `StudyInstanceUID`/`Report` columns but never carries targets — Phase 3A predicts them. `train_series.csv` and `test_series.csv` hold per-series acquisition metadata (no report text), and `sample_submission.csv` defines the exact submission schema. The schema view above shows *types*, not values — `StudyInstanceUID` and `Report` are never displayed here in raw form, and the 12 target columns appear only as aggregate prevalence in Section 3. The protocol-combination table reports every observed (plane, fluid-sensitivity, fat-suppression) combination as an aggregate series count and share — no individual series or study identifier is shown anywhere in this notebook.

## 3. Human Labels and Target Prevalence

In [ ]:
positive_counts = labeled[LABEL_COLUMNS].sum().astype(int)
positive_rates = labeled[LABEL_COLUMNS].mean()
prevalence_table = (
    pd.DataFrame(
        {
            "Positive Studies": positive_counts,
            "Positive Rate": positive_rates,
        }
    )
    .sort_values("Positive Rate", ascending=False)
)
display(prevalence_table)

fig, ax = plt.subplots(figsize=(9, 5))
prevalence_table["Positive Rate"].sort_values().plot(
    kind="barh",
    ax=ax,
    color="#2878B5",
)
ax.set_xlabel("Positive Rate Among 58 Human-Labeled Studies")
ax.set_ylabel("")
ax.set_title("Human-Label Prevalence Across 12 Targets")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

**Interpretation.** Target prevalence varies materially across the
12 abnormalities, so one aggregate score can hide label-specific behavior.
The labeled sample is also too small for repeated tuning: later modeling
reports pooled macro AUC together with per-label and fold diagnostics.

## 4. MRI Series Composition

In [ ]:
series_per_study = series_df.groupby("StudyInstanceUID").size()
series_summary = series_per_study.describe(
    percentiles=[0.25, 0.5, 0.75]
).rename("Series per Study").to_frame()
plane_counts = (
    series_df["Anatomical_Plane"]
    .value_counts()
    .rename_axis("Anatomical Plane")
    .to_frame("Series")
)
sequence_summary = (
    series_df[["Fluid_Sensitive", "Fat_Suppression"]]
    .mean()
    .rename("Share of Series")
    .to_frame()
)
display(series_summary)
display(plane_counts)
display(sequence_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
series_per_study.plot(kind="hist", bins=30, ax=axes[0], color="#2878B5")
axes[0].set_xlabel("Series per Study")
axes[0].set_title("Series Count Distribution")
plane_counts["Series"].plot(kind="bar", ax=axes[1], color="#4C956C")
axes[1].set_xlabel("")
axes[1].set_ylabel("Series")
axes[1].set_title("Series by Anatomical Plane")
for axis in axes:
    axis.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

**Interpretation.** The 24,371 series span all three anatomical
planes: 9,864 sagittal, 8,609 coronal, and 5,898 axial. Sagittal is most
common, but no plane is rare. Fluid-sensitive and fat-suppression flags both
average 0.5749; equal means alone do not establish that the flags are
identical for every series.

## 5. Aggregate Report Characteristics

In [ ]:
report_lengths = train_df["Report"].fillna("").str.len()
report_summary = report_lengths.describe(
    percentiles=[0.25, 0.5, 0.75]
).rename("Characters per Report").to_frame()

report_buckets = train_df["Report"].fillna("").map(orthographic_bucket)
bucket_counts = report_buckets.value_counts()
orthographic_counts = pd.DataFrame(
    {
        "Studies": bucket_counts,
        "Share": bucket_counts / bucket_counts.sum(),
    }
)
display(report_summary)
display(orthographic_counts)

**Interpretation.** Report length varies across the training set,
and the aggregate character-set buckets confirm meaningful orthographic
variation. These buckets describe observed scripts and diacritics; they are
not language identification and do not establish why any model succeeds or
fails. Character n-grams are consequently a more defensible first text
representation than a small English-only word vocabulary.

## 6. Slice-Count Distribution

In [ ]:
train_series_root = DATA_DIR / "train_series"
study_directories = sorted(
    path for path in train_series_root.iterdir() if path.is_dir()
)[:200]
slice_counts = []
for study_directory in study_directories:
    for series_directory in sorted(study_directory.iterdir()):
        if series_directory.is_dir():
            slice_counts.append(len(tuple(series_directory.glob("*.dcm"))))

slice_counts = pd.Series(slice_counts, dtype="int64")
slice_summary = slice_counts.describe(
    percentiles=[0.25, 0.5, 0.75]
).rename("Slices per Series").to_frame()
display(slice_summary)

fig, ax = plt.subplots(figsize=(9, 4))
slice_counts.plot(kind="hist", bins=30, ax=ax, color="#F28E2B")
ax.set_xlabel("Slices per Series")
ax.set_title("Slice Counts Across the First 200 Studies")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

**Interpretation.** The deterministic first-200-study scan covers
1,109 series, with median 30 slices and a long tail reaching 320. It is a
runtime-conscious structural check rather than a random or population-
representative sample, so it supports pipeline planning but not distributional
inference about every series.

## 7. Findings, Limitations, and Modeling Implications

- Only 58 of 4,407 studies have complete human labels; the other 4,349 are
  excluded from supervised Phase 3A fitting.
- All three MRI planes are well represented across 24,371 series, while
  per-study series and slice counts have meaningful spread and long tails.
- Aggregate report evidence shows varied scripts and diacritics. The coarse
  buckets are descriptive, not language labels or causal explanations.
- Label scarcity makes leakage-safe internal cross-validation essential and
  makes extensive hyperparameter searching unreliable.
- Phase 3A establishes a transparent report-only baseline. Image information
  remains deliberately deferred to a future Phase 3B imaging baseline.